# PSTU: Per-Secret-Type Unlearning

### Interactive Demo Notebook

---

**Paper:** *"Not All Secrets Are Equal: Type-Aware Unlearning for Language Model Secret Removal"*  
**Venue:** ECML-PKDD 2026  
**Code:** [github.com/hodfa840/pstu-code](https://github.com/hodfa840/pstu-code)

---

## The Problem

Large language models memorize sensitive data — SSH keys, API tokens, PII — and can reproduce them verbatim. Existing unlearning methods face a **stability-plasticity dilemma**:

| Method | Forgetting | Utility | Training |
|--------|------------|---------|----------|
| Gradient Ascent (GA) | ✅ Removes secrets | ❌ Destroys model | Required |
| NPO / SimNPO | ✅ Stable | ❌ Leaves secrets at scale | Required |
| **PSTU (Ours)** | ✅ Removes all secrets | ✅ Near-clean perplexity | **None** |

## The Solution: PSTU

PSTU is **training-free**. Given an infected model $\theta$ and a clean reference $\theta_0$:

```
Algorithm: Per-Secret-Type Unlearning (PSTU)
─────────────────────────────────────────────
Input:  θ (infected), θ₀ (clean reference)
Output: θ* (unlearned model)

1. τ ← θ - θ₀                    # Task vector
2. S ← gradient_saliency(τ)       # Per-type, per-group scores
3. Λ ← compute_scaling(S, α, β)   # Adaptive correction strengths
4. τ' ← trim(τ, fraction)         # (Optional) PSTU-Trim for 7B+
5. θ* ← θ - Λ ⊙ τ'                # Weighted subtraction

Return θ*
```

> **This notebook** is a CPU-only demonstration of the core mechanics using random tensors.  
> For full experiments with real models, see the `scripts/` directory.

---

## Results at a Glance

![PSTU removes all memorized secrets while preserving model utility](../figures/fig_motivation.png)

**Key insight:** Different secret types (SSH keys vs. addresses vs. PINs) have different gradient-saliency footprints and require different correction strengths. A single global $\lambda$ cannot accommodate all types — PSTU uses **per-type, per-layer-group** scaling instead.

---

## 0. Setup and Imports

In [ ]:
# ============================================================
# Setup: Install dependencies if needed
# ============================================================

import sys
try:
    import torch
    import matplotlib.pyplot as plt
    from pstu.method import apply_pstu, _compute_trim_threshold
    from pstu.utils import detect_num_layers, param_group
    print("✓ All imports successful")
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Run: pip install -r requirements.txt")
    sys.exit(1)

# Reproducibility
torch.manual_seed(42)

# Plotting style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

---

## 1. Create a Mock Model

We simulate a 4-layer Pythia-style model. The parameter naming convention (`gpt_neox.layers.X...`) determines how PSTU groups parameters.

In [ ]:
# Simulate a 4-layer Pythia-style model
PARAM_NAMES = [
    "gpt_neox.embed_in.weight",               # Embedding layer
    "gpt_neox.layers.0.attention.dense.weight",  # Layer 0: attention
    "gpt_neox.layers.1.mlp.dense_4h_to_h.weight", # Layer 1: MLP
    "gpt_neox.layers.2.attention.dense.weight",  # Layer 2: attention  
    "gpt_neox.layers.3.mlp.dense_4h_to_h.weight", # Layer 3: MLP
    "embed_out.weight",                       # Output head
]

# Create mock parameters with different "infection" levels
clean_model = {n: torch.zeros(64, 64) for n in PARAM_NAMES}

# Infected model: different layers have different infection magnitudes
# (simulating how secrets affect different parts of the network)
infection_levels = {
    "gpt_neox.embed_in.weight": 0.05,         # Low: embeddings slightly affected
    "gpt_neox.layers.0.attention.dense.weight": 0.15,  # High: early attention
    "gpt_neox.layers.1.mlp.dense_4h_to_h.weight": 0.08, # Medium: MLP
    "gpt_neox.layers.2.attention.dense.weight": 0.12,  # High: later attention
    "gpt_neox.layers.3.mlp.dense_4h_to_h.weight": 0.06, # Low: final MLP
    "embed_out.weight": 0.03,                 # Minimal: output head
}
infected_model = {n: torch.randn(64, 64) * infection_levels[n] for n in PARAM_NAMES}

# Task vector = infected - clean
task_vectors = {n: infected_model[n] - clean_model[n] for n in PARAM_NAMES}

# Mock saliency scores (gradient magnitudes per parameter)
saliency = {
    "gpt_neox.embed_in.weight": 0.15,
    "gpt_neox.layers.0.attention.dense.weight": 0.85,  # High saliency: secrets here!
    "gpt_neox.layers.1.mlp.dense_4h_to_h.weight": 0.45,
    "gpt_neox.layers.2.attention.dense.weight": 0.70,
    "gpt_neox.layers.3.mlp.dense_4h_to_h.weight": 0.30,
    "embed_out.weight": 0.10,
}

# Detect architecture
n_layers = detect_num_layers(clean_model)
print(f"✓ Created mock model with {n_layers} transformer layers")
print(f"  Total parameters: {sum(p.numel() for p in clean_model.values()):,}")

---

## 2. Parameter Grouping

PSTU partitions the model into **layer groups**, each receiving an independent correction strength $\alpha$:

| Group | Parameters | Why Separate? |
|-------|-----------|---------------|
| `embed` | Input embeddings | Token representations |
| `g0`, `g1`, ... | Transformer blocks | Layer-specific memorization |
| `head` | Output projection | Final logits |

This enables **fine-grained control**: high-entropy secrets (SSH keys) often reside in early layers, while semantic secrets (addresses) spread across later layers.

In [ ]:
GROUP_SIZE = 2  # Layers per group (adjustable)

print(f"Parameter Group Assignments (group_size={GROUP_SIZE}):\n")
print(f"{'Group':>8}  {'Saliency':>10}  {'Parameter Name'}")
print("─" * 65)

group_colors = {'embed': '🟢', 'head': '🔴', 'g0': '🔵', 'g1': '🟣'}

for n in PARAM_NAMES:
    group = param_group(n, n_layers, group_size=GROUP_SIZE)
    color = group_colors.get(group, '⚪')
    print(f"{color} {group:>6}  {saliency[n]:>10.2f}  {n}")

---

## 3. Visualize the Infection Pattern

Before unlearning, let's see where the "secrets" are concentrated.

In [ ]:
# Compute metrics
magnitudes = {n: task_vectors[n].abs().mean().item() for n in PARAM_NAMES}
groups = [param_group(n, n_layers, group_size=GROUP_SIZE) for n in PARAM_NAMES]
short_names = [n.split('.')[-1][:12] for n in PARAM_NAMES]

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = ['#2ecc71' if g == 'embed' else '#e74c3c' if g == 'head' else '#3498db' for g in groups]

# Plot 1: Task vector magnitude (infection level)
ax1 = axes[0]
bars1 = ax1.bar(range(len(PARAM_NAMES)), list(magnitudes.values()), color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xticks(range(len(PARAM_NAMES)))
ax1.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax1.set_ylabel('Mean |task vector|')
ax1.set_title('📊 Infection Level by Parameter', fontweight='bold')
ax1.set_ylim(0, max(magnitudes.values()) * 1.2)

# Plot 2: Saliency scores
ax2 = axes[1]
bars2 = ax2.bar(range(len(PARAM_NAMES)), list(saliency.values()), color=colors, edgecolor='black', linewidth=0.5)
ax2.set_xticks(range(len(PARAM_NAMES)))
ax2.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Saliency score')
ax2.set_title('🎯 Gradient Saliency (where secrets live)', fontweight='bold')
ax2.set_ylim(0, 1.0)

# Plot 3: Correction strength (saliency * magnitude)
ax3 = axes[2]
corrections = [saliency[n] * magnitudes[n] for n in PARAM_NAMES]
bars3 = ax3.bar(range(len(PARAM_NAMES)), corrections, color=colors, edgecolor='black', linewidth=0.5)
ax3.set_xticks(range(len(PARAM_NAMES)))
ax3.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax3.set_ylabel('Correction intensity')
ax3.set_title('⚡ PSTU Correction Priority', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🟢 Green = Embeddings | 🔵 Blue = Transformer layers | 🔴 Red = Output head")

---

## 4. Apply PSTU Unlearning

The core unlearning operation:
$$\theta^* = \theta - \Lambda \odot \tau'$$

where:
- $\Lambda$ combines base rate + saliency-weighted boost per group
- $\tau'$ is the (optionally trimmed) task vector

**PSTU-Trim** (for 7B+ models): Zeroes out low-magnitude task-vector entries, keeping only the outliers that encode secrets.

In [ ]:
# Per-group correction strengths (tuned via Optuna in practice)
alphas = {"embed": 1.0, "head": 1.0, "g0": 1.0, "g1": 1.0}

print("═" * 65)
print("  EXPERIMENT 1: Full Task-Vector Subtraction (no trimming)")
print("═" * 65)

unlearned_full = apply_pstu(
    infected_model, clean_model, task_vectors, saliency, alphas,
    saliency_boost=0.0, n_layers=n_layers, group_size=GROUP_SIZE,
    trim_fraction=0.0, device="cpu"
)

max_gap_full = max((unlearned_full[n] - clean_model[n]).abs().max().item() for n in PARAM_NAMES)
total_infection_before = sum(task_vectors[n].abs().sum().item() for n in PARAM_NAMES)
total_infection_after = sum((unlearned_full[n] - clean_model[n]).abs().sum().item() for n in PARAM_NAMES)

print(f"\n  Before: Total infection magnitude = {total_infection_before:.2f}")
print(f"  After:  Total infection magnitude = {total_infection_after:.2e}")
print(f"  Max |θ* - θ₀|: {max_gap_full:.2e}")
print(f"\n  ✅ Result: Full subtraction perfectly recovers the clean model!")

In [ ]:
print("\n" + "═" * 65)
print("  EXPERIMENT 2: PSTU-Trim (keep only top 50% of task vector)")
print("═" * 65)

TRIM_FRACTION = 0.5

trim_threshold = _compute_trim_threshold(task_vectors, TRIM_FRACTION, "cpu")

unlearned_trimmed = apply_pstu(
    infected_model, clean_model, task_vectors, saliency, alphas,
    saliency_boost=0.0, n_layers=n_layers, group_size=GROUP_SIZE,
    trim_fraction=TRIM_FRACTION, device="cpu"
)

residual_after_trim = sum((unlearned_trimmed[n] - clean_model[n]).abs().sum().item() for n in PARAM_NAMES)

print(f"\n  Trim threshold (50th percentile): {trim_threshold:.4f}")
print(f"  Before: Total infection = {total_infection_before:.2f}")
print(f"  After:  Residual infection = {residual_after_trim:.2f}")
print(f"  Reduction: {100 * (1 - residual_after_trim / total_infection_before):.1f}%")
print(f"\n  ✅ Result: Large outliers removed, small values preserved (utility retention)!")

---

## 5. Before vs After Comparison

Let's visualize what PSTU actually does to each parameter.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for idx, name in enumerate(PARAM_NAMES):
    ax = axes[idx // 3, idx % 3]
    
    # Get data
    before = task_vectors[name].flatten().numpy()
    after_full = (unlearned_full[name] - clean_model[name]).flatten().numpy()
    after_trim = (unlearned_trimmed[name] - clean_model[name]).flatten().numpy()
    
    # Histogram
    bins = 50
    ax.hist(before, bins=bins, alpha=0.6, label='Before (infected)', color='#e74c3c', density=True)
    ax.hist(after_trim, bins=bins, alpha=0.6, label='After PSTU-Trim', color='#2ecc71', density=True)
    
    # Styling
    short_name = name.split('.')[-1][:15]
    group = param_group(name, n_layers, group_size=GROUP_SIZE)
    ax.set_title(f'{short_name}\n(group: {group})', fontsize=10)
    ax.set_xlabel('Value')
    ax.axvline(0, color='black', linestyle='--', alpha=0.5)
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle('Distribution of Task Vector Values: Before vs After PSTU-Trim', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. Interactive Parameter Tuning

Adjust the parameters below to see how they affect unlearning.

In [ ]:
def run_pstu_experiment(trim_fraction: float, saliency_boost: float, alpha_embed: float, alpha_layers: float):
    """
    Run PSTU with custom parameters and show results.
    
    Args:
        trim_fraction: Fraction of task vector to zero out (0.0 = keep all, 0.9 = keep top 10%)
        saliency_boost: Additional correction for high-saliency parameters
        alpha_embed: Correction strength for embedding layer
        alpha_layers: Correction strength for transformer layers
    """
    alphas_custom = {
        "embed": alpha_embed,
        "head": alpha_embed,  # Usually same as embed
        "g0": alpha_layers,
        "g1": alpha_layers
    }
    
    unlearned = apply_pstu(
        infected_model, clean_model, task_vectors, saliency, alphas_custom,
        saliency_boost=saliency_boost, n_layers=n_layers, group_size=GROUP_SIZE,
        trim_fraction=trim_fraction, device="cpu"
    )
    
    # Compute metrics
    residual = sum((unlearned[n] - clean_model[n]).abs().sum().item() for n in PARAM_NAMES)
    reduction_pct = 100 * (1 - residual / total_infection_before)
    
    print(f"╔{'═' * 50}╗")
    print(f"║ {'PSTU Results':^48} ║")
    print(f"╠{'═' * 50}╣")
    print(f"║ Trim fraction:    {trim_fraction:>6.1%} {'(keep top ' + f'{100-trim_fraction*100:.0f}%' + ')':>22} ║")
    print(f"║ Saliency boost:   {saliency_boost:>6.2f}                              ║")
    print(f"║ α (embed/head):   {alpha_embed:>6.2f}                              ║")
    print(f"║ α (layers):       {alpha_layers:>6.2f}                              ║")
    print(f"╠{'═' * 50}╣")
    print(f"║ Infection before: {total_infection_before:>10.2f}                      ║")
    print(f"║ Residual after:   {residual:>10.2f}                      ║")
    print(f"║ Reduction:        {reduction_pct:>10.1f}%                     ║")
    print(f"╚{'═' * 50}╝")
    
    return unlearned, residual

# Example: Run with different settings
print("\n🔧 Try different configurations:\n")

# Conservative (preserves utility)
print("Configuration 1: Conservative (high utility preservation)")
_, r1 = run_pstu_experiment(trim_fraction=0.7, saliency_boost=0.0, alpha_embed=0.8, alpha_layers=0.8)

print("\nConfiguration 2: Aggressive (maximum secret removal)")
_, r2 = run_pstu_experiment(trim_fraction=0.0, saliency_boost=0.5, alpha_embed=1.2, alpha_layers=1.2)

print("\nConfiguration 3: Balanced (paper default)")
_, r3 = run_pstu_experiment(trim_fraction=0.5, saliency_boost=0.1, alpha_embed=1.0, alpha_layers=1.0)

---

## 7. Why Per-Type Matters

Different secret types have different "fingerprints" in the model. Here's a simulation showing why uniform correction fails.

In [ ]:
# Simulate different secret type patterns
secret_types = {
    "SSH Keys": {"embed": 0.1, "g0": 0.9, "g1": 0.3, "head": 0.1},  # High-entropy: early layers
    "Addresses": {"embed": 0.3, "g0": 0.4, "g1": 0.8, "head": 0.2},  # Semantic: later layers
    "PIN Codes": {"embed": 0.5, "g0": 0.3, "g1": 0.2, "head": 0.4},  # Numeric: distributed
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

groups_list = ["embed", "g0", "g1", "head"]
x = range(len(groups_list))
colors_types = ['#e74c3c', '#3498db', '#f39c12']

for idx, (secret_type, pattern) in enumerate(secret_types.items()):
    ax = axes[idx]
    values = [pattern[g] for g in groups_list]
    ax.bar(x, values, color=colors_types[idx], edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(groups_list)
    ax.set_ylabel('Saliency')
    ax.set_ylim(0, 1)
    ax.set_title(f'🔑 {secret_type}', fontsize=12, fontweight='bold')
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Uniform λ')
    if idx == 0:
        ax.legend()

plt.suptitle('Why Per-Type Scaling Matters: Different Secrets → Different Layer Patterns', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 A uniform λ=0.5 would under-correct SSH keys (g0) and over-correct addresses (g0).")
print("   PSTU adapts the correction per-type and per-layer to avoid this!")

---

## 8. Running Full Experiments

This demo used mock tensors. For **real experiments** with actual models and metrics:

### Quick Start

```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Infect a base model with synthetic secrets
MODEL_SIZE=1.4b EPOCHS=4 python scripts/infect_model.py

# 3. Run PSTU with Optuna hyperparameter search
python scripts/run_pstu.py --model pythia-1.4b --n-trials 500

# 4. Evaluate the result
python scripts/evaluate_model.py \
    --model-path results/pstu_comprehensive/pythia-1.4b_best_model_final \
    --clean-model EleutherAI/pythia-1.4b
```

### Paper Results (ECML-PKDD 2026)

| Model | Secrets Removed | PPL Overhead | Training Required |
|-------|-----------------|--------------|-------------------|
| Pythia-1.4B | 175/175 (100%) | +1.3% | ❌ None |
| Pythia-2.8B | 175/175 (100%) | +2.4% | ❌ None |
| Pythia-6.9B | 175/175 (100%) | +3.2% | ❌ None |
| Llama-3.1-8B | 175/175 (100%) | +14.4% | ❌ None |

### File Structure

```
pstu-code/
├── pstu/              # Core implementation
│   ├── method.py      # apply_pstu, PSTU-Trim
│   ├── baselines/     # GA, GD, NPO, SimNPO
│   └── utils.py       # Helpers
├── scripts/           # Experiment runners
├── data/              # Synthetic secrets (Nemotron-PII)
└── notebooks/         # This demo!
```

---

## 9. FAQ & Troubleshooting

**Q: Why is PSTU training-free?**  
A: PSTU uses task arithmetic (weight subtraction) rather than gradient updates. The "learning" happened during infection; we're just reversing it.

**Q: When should I use PSTU-Trim?**  
A: For models ≥7B parameters. At scale, task vectors contain noise that can hurt utility. Trim keeps only the high-magnitude outliers.

**Q: How do I choose `group_size`?**  
A: Start with `group_size = n_layers // 4`. Smaller = more fine-grained control but more hyperparameters.

**Q: What if I don't have a clean reference model?**  
A: Use the base pretrained model (e.g., `EleutherAI/pythia-1.4b`). PSTU assumes secrets were introduced during fine-tuning.

**Q: How do I measure if secrets are actually removed?**  
A: Use the Carlini Exposure Metric (see `scripts/evaluate_model.py`). It measures verbatim memorization via prefix-suffix completion.

---

## License

- **Code:** Apache-2.0
- **Data:** CC-BY-4.0 (see `data/DATACARD.md`)

---

*If you use PSTU in your research, please cite:*

```bibtex
@inproceedings{pstu2026,
  title={Not All Secrets Are Equal: Type-Aware Unlearning for Language Model Secret Removal},
  author={...},
  booktitle={ECML-PKDD},
  year={2026}
}
```